# Kaggle submission

Обучим финальную модель на всём train и получим предсказания для test. Используем выбранные признаки, параметры trial 13 и 1758 деревьев.

In [1]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier

pd.set_option("display.max_columns", 100)

RANDOM_STATE = 42
TARGET = "isFraud"
ID_COLUMN = "TransactionID"
SECONDS_PER_DAY = 86_400
FINAL_N_ESTIMATORS = 1758

project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent
data_dir = project_root / "data"
submission_dir = project_root / "submissions"

## 1. Признаки

Train и test должны содержать одинаковые признаки. В test названия `id-01` приводим к формату train: `id_01`.

In [2]:
transaction_columns = [
    "TransactionID", "TransactionDT", "TransactionAmt", "ProductCD",
    "card1", "card2", "card3", "card4", "card5", "card6",
    "addr1", "addr2", "dist1", "dist2",
    "P_emaildomain", "R_emaildomain",
    *[f"C{i}" for i in range(1, 15)],
    *[f"D{i}" for i in range(1, 16)],
    *[f"M{i}" for i in range(1, 10)],
]

selected_v_columns = (
    "V281 V282 V283 V289 V296 V301 V314 V1 V3 V4 V6 V8 V11 V13 V14 V17 V20 "
    "V23 V26 V27 V30 V36 V37 V40 V41 V44 V47 V48 V54 V55 V56 V59 V62 V65 V67 "
    "V68 V70 V76 V78 V80 V82 V86 V88 V89 V91 V127 V98 V99 V107 V108 V109 V111 "
    "V114 V115 V117 V120 V121 V123 V129 V130 V131 V138 V139 V142 V147 V155 V162 "
    "V165 V160 V166 V203 V207 V173 V169 V171 V174 V175 V180 V185 V198 V210 V209 "
    "V264 V223 V240 V241 V260 V277 V220 V221 V234 V238 V250 V271 V307 V284 V285 "
    "V286 V291 V303 V305 V309 V310 V312 V320 V332 V325 V335"
).split()

model_transaction_columns = transaction_columns + selected_v_columns

categorical_columns = [
    "ProductCD", "card4", "card6", "P_emaildomain", "R_emaildomain",
    *[f"M{i}" for i in range(1, 10)],
    "id_12", "id_15", "id_16", "id_23", "id_27", "id_28",
    "id_29", "id_30", "id_31", "id_33", "id_34", "id_35",
    "id_36", "id_37", "id_38", "DeviceType", "DeviceInfo",
]

identity_columns = [
    *[f"id_{i:02d}" for i in range(1, 10)],
    *[f"id_{i}" for i in range(10, 39)],
]

train_dtypes = {
    column: "float32"
    for column in model_transaction_columns + identity_columns
}
train_dtypes.update({column: "category" for column in categorical_columns})
train_dtypes[ID_COLUMN] = "int32"
train_dtypes[TARGET] = "int8"

test_identity_dtypes = {
    column.replace("_", "-"): dtype
    for column, dtype in train_dtypes.items()
    if column.startswith("id_")
}
test_identity_dtypes.update({
    "TransactionID": "int32",
    "DeviceType": "category",
    "DeviceInfo": "category",
})

selected_d_columns = ["D1", "D2", "D4", "D10", "D11", "D15"]

## 2. Загрузка train и test

Объединяем transaction и identity по `TransactionID`.

In [3]:
train_transactions = pd.read_csv(
    data_dir / "train_transaction.csv",
    index_col=ID_COLUMN,
    usecols=model_transaction_columns + [TARGET],
    dtype=train_dtypes,
)
train_identity = pd.read_csv(
    data_dir / "train_identity.csv",
    index_col=ID_COLUMN,
    dtype=train_dtypes,
)

y_train = train_transactions.pop(TARGET).to_numpy(dtype=np.int8)
X_train = train_transactions.join(train_identity, how="left")

del train_transactions, train_identity

print("Train:", X_train.shape)
print("Train fraud rate:", f"{y_train.mean():.2%}")

Train: (590540, 201)
Train fraud rate: 3.50%


In [4]:
test_transactions = pd.read_csv(
    data_dir / "test_transaction.csv",
    index_col=ID_COLUMN,
    usecols=model_transaction_columns,
    dtype=train_dtypes,
)
test_identity = pd.read_csv(
    data_dir / "test_identity.csv",
    index_col=ID_COLUMN,
    dtype=test_identity_dtypes,
)
test_identity.columns = test_identity.columns.str.replace("-", "_", regex=False)

X_test = test_transactions.join(test_identity, how="left")

del test_transactions, test_identity


print("Test:", X_test.shape)

Test: (506691, 201)


## 3. Feature engineering

Повторяем преобразования финальной модели. Encoder обучается на train, затем применяется к test.

In [5]:
def add_time_features(data):
    df = data.copy()

    df["hour"] = (df["TransactionDT"] // 3_600) % 24
    df["weekday"] = (df["TransactionDT"] // SECONDS_PER_DAY) % 7

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["weekday_sin"] = np.sin(2 * np.pi * df["weekday"] / 7)
    df["weekday_cos"] = np.cos(2 * np.pi * df["weekday"] / 7)

    return df


def normalize_selected_d_features(data, columns=selected_d_columns):
    df = data.copy()
    transaction_day = df["TransactionDT"] / SECONDS_PER_DAY

    for column in columns:
        df[column] = (df[column] - transaction_day).astype("float32")

    return df


def build_features(data):
    return normalize_selected_d_features(add_time_features(data))

In [6]:
feature_started = perf_counter()

X_train_features = build_features(X_train)
X_test_features = build_features(X_test)

del X_train, X_test

categorical_model_columns = [
    column for column in categorical_columns
    if column in X_train_features.columns
]
numeric_model_columns = [
    column for column in X_train_features.columns
    if column not in categorical_model_columns
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1,
                encoded_missing_value=-2,
                dtype=np.float32,
            ),
            categorical_model_columns,
        ),
        ("numeric", "passthrough", numeric_model_columns),
    ],
    verbose_feature_names_out=False,
)

X_train_matrix = preprocessor.fit_transform(X_train_features)
X_test_matrix = preprocessor.transform(X_test_features)

X_train_matrix = X_train_matrix.astype(np.float32, copy=False)
X_test_matrix = X_test_matrix.astype(np.float32, copy=False)

print("Train matrix:", X_train_matrix.shape)
print("Test matrix:", X_test_matrix.shape)
print(f"Feature preparation: {perf_counter() - feature_started:.1f}s")

del X_train_features, X_test_features

Train matrix: (590540, 207)
Test matrix: (506691, 207)
Feature preparation: 11.9s


## 4. Обучение

Обучаем модель на полном train без early stopping.

In [7]:
final_params = {
    "n_estimators": FINAL_N_ESTIMATORS,
    "learning_rate": 0.015442878662368152,
    "max_depth": 7,
    "min_child_weight": 3.3973837770378004,
    "subsample": 0.8549473812230148,
    "colsample_bytree": 0.7161707124793311,
    "gamma": 0.06801678402593356,
    "reg_alpha": 0.002929317375402625,
    "reg_lambda": 14.622376387551679,
    "eval_metric": "auc",
    "tree_method": "hist",
    "max_bin": 256,
    "verbosity": 0,
    "n_jobs": -1,
    "random_state": RANDOM_STATE,
}

final_model = XGBClassifier(**final_params)
display(pd.Series(final_params, name="value").to_frame())

,value
n_estimators,1758
learning_rate,0.015443
max_depth,7
min_child_weight,3.397384
subsample,0.854947
colsample_bytree,0.716171
gamma,0.068017
reg_alpha,0.002929
reg_lambda,14.622376
eval_metric,auc


In [8]:
fit_started = perf_counter()
final_model.fit(X_train_matrix, y_train, verbose=False)
fit_seconds = perf_counter() - fit_started

print(f"Fit time: {fit_seconds / 60:.1f} min")

Fit time: 1.3 min


## 5. Submission

Получаем вероятности fraud и записываем их в формат `sample_submission.csv`.

In [9]:
test_proba = final_model.predict_proba(X_test_matrix)[:, 1]

prediction_summary = pd.Series(test_proba).describe(
    percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
)
display(prediction_summary.to_frame("fraud_probability"))

,fraud_probability
count,506691.000000
mean,0.037962
std,0.118267
min,0.000144
1%,0.000891
5%,0.001634
50%,0.009562
95%,0.126103
99%,0.855754
max,0.998445


In [10]:
sample_submission = pd.read_csv(
    data_dir / "sample_submission.csv",
    dtype={ID_COLUMN: "int32", TARGET: "float32"},
)


submission = sample_submission.copy()
submission[TARGET] = test_proba


submission_dir.mkdir(parents=True, exist_ok=True)
submission_path = submission_dir / "xgboost_trial13_full_train.csv"
submission.to_csv(submission_path, index=False)

print("Saved:", submission_path)
print(f"File size: {submission_path.stat().st_size / 1024**2:.1f} MB")
display(submission.head())

Saved: C:\Users\Fedor\Projects\antifraud\submissions\xgboost_trial13_full_train.csv
File size: 10.2 MB


,TransactionID,isFraud
0,3663549,0.003712
1,3663550,0.004944
2,3663551,0.014907
3,3663552,0.003305
4,3663553,0.003754


## 6. Результат

Submission получил ROC-AUC 0.937537 на Public и 0.911611 на Private leaderboard. Private Score близок к локальному holdout 0.90913.